# Weak-Lensing Daten & Labels — Visualisierung

Dieses Notebook zeigt, **wie die Labels und die κ-Karten** des FAIR-Universe
Weak-Lensing-Datensatzes aussehen (derselbe Datensatz wie in
`src/dataset.py` / `WeakLensingDataset`).

Aufbau:
1. **Setup** — Libraries installieren & Datenpfad setzen
2. **Labels** — Kosmologie-Gitter $(\Omega_m, S_8)$, Verteilungen, Struktur
3. **Daten** — Maske, κ-Beispielkarten, Effekt des Shape-Noise, Pixelverteilung

> **Hinweis:** Die κ-Datei ist mehrere GB groß und liegt auf dem Cluster.
> Das Notebook lädt sie per `mmap` (nur einzelne Karten landen im RAM) und nutzt
> einen konfigurierbaren `DATA_DIR`.

## 1. Setup

In [ ]:
# Benötigte Libraries direkt im Notebook installieren.
%pip install -q numpy matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "white" 

### Datenpfad konfigurieren

Default ist der Cluster-Pfad aus `configs/dataset/weak_lensing.yaml`. Lokal
entweder hier anpassen oder die Umgebungsvariable `WL_DATA_DIR` setzen.

In [ ]:
# Pfad zu den .npy-Dateien des FAIR-Universe Weak-Lensing-Datensatzes.
DATA_DIR = os.environ.get("WL_DATA_DIR", "/cluster/projects/ska/weak-lensing/data/")

KAPPA_FILE = "WIDE12H_bin2_2arcmin_kappa.npy"
LABEL_FILE = "label.npy"
MASK_FILE  = "WIDE12H_bin2_2arcmin_mask.npy"
TEST_FILE  = "WIDE12H_bin2_2arcmin_kappa_noisy_test.npy"

NCOSMO, NSYS   = 101, 256     # 101 Kosmologien x 256 Systematik-/Noise-Realisierungen
IMG_H, IMG_W   = 1424, 176    # Kartendimension (hoch & schmal)
NG, PIX_ARCMIN = 30.0, 2.0    # Galaxiendichte / Pixelgröße -> Shape-Noise

assert os.path.isdir(DATA_DIR), (
    f"DATA_DIR nicht gefunden: {DATA_DIR}\n"
    "Setze die Umgebungsvariable WL_DATA_DIR oder passe den Pfad oben an."
)
print("Daten-Verzeichnis:", DATA_DIR)
print("Vorhandene .npy-Dateien:", sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".npy")))

### Maske, Labels und κ-Karten laden

In [ ]:
# Maske: bool (H, W). Gespeichert werden nur die unmaskierten Pixel pro Karte.
mask = np.load(os.path.join(DATA_DIR, MASK_FILE)).astype(bool)
n_unmasked = int(mask.sum())
print(f"Maske: {mask.shape}, unmaskierte Pixel: {n_unmasked} "
      f"({100 * n_unmasked / mask.size:.1f}% von {mask.size})")

# Labels: (Ncosmo, Nsys, 2) -> [Omega_m, S8]
labels = np.load(os.path.join(DATA_DIR, LABEL_FILE)).astype(np.float32)
print("Labels:", labels.shape, "-> (Ncosmo, Nsys, [Omega_m, S8])")

# kappa als memmap (gesamtes Array mehrere GB -> NICHT in den RAM ziehen).
kappa = np.load(os.path.join(DATA_DIR, KAPPA_FILE), mmap_mode="r")
kappa = kappa.reshape(NCOSMO, NSYS, n_unmasked)   # View auf das memmap
print("kappa (compact, memmap):", kappa.shape, kappa.dtype)

## 2. Labels

Der Datensatz besteht aus **101 Kosmologien**, jede mit eigenem Paar
$(\Omega_m, S_8)$. Pro Kosmologie gibt es **256 Realisierungen** (verschiedene
systematische/Noise-Seeds, aber *dieselbe* Kosmologie). Wir prüfen das zuerst
und visualisieren dann das Parameter-Gitter.

In [ ]:
# Variieren die Labels entlang der Systematik-Achse (axis=1)? Erwartung: nein.
std_along_sys = labels.std(axis=1)            # (Ncosmo, 2)
print("max Label-Std entlang Nsys (Om, S8):", std_along_sys.max(axis=0),
      "-> ~0 bedeutet: 256 Realisierungen teilen dieselbe Kosmologie")

# Eindeutige Kosmologie = erstes Label je Ncosmo.
cosmo = labels[:, 0, :]                        # (Ncosmo, 2)
Om, S8 = cosmo[:, 0], cosmo[:, 1]
print(f"Omega_m in [{Om.min():.3f}, {Om.max():.3f}]   "
      f"S8 in [{S8.min():.3f}, {S8.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

sc = axes[0].scatter(Om, S8, c=np.arange(NCOSMO), cmap="viridis", s=40, edgecolor="k", linewidth=0.3)
axes[0].set(xlabel=r"$\Omega_m$", ylabel=r"$S_8$",
            title=f"Kosmologie-Gitter ({NCOSMO} Knoten)")
fig.colorbar(sc, ax=axes[0], label="Kosmologie-Index")

axes[1].hist(Om, bins=25, color="steelblue", edgecolor="white")
axes[1].set(xlabel=r"$\Omega_m$", ylabel="Anzahl Kosmologien", title=r"Verteilung $\Omega_m$")

axes[2].hist(S8, bins=25, color="indianred", edgecolor="white")
axes[2].set(xlabel=r"$S_8$", ylabel="Anzahl Kosmologien", title=r"Verteilung $S_8$")

plt.tight_layout()
plt.show()

In [ ]:
# So sieht die score/ensemble-Pipeline die Labels: z-standardisiert (vgl. dataset.py).
mu, sd = cosmo.mean(axis=0), cosmo.std(axis=0)
print("Label-Mittelwert (Om, S8):", mu)
print("Label-Std        (Om, S8):", sd)
print("=> standardisiert leben Om und S8 auf einer O(1)-Skala (wichtig für die score-Loss).")

## 3. Daten (κ-Karten)

Die Karten sind **1424×176** — hoch und schmal. Jede gespeicherte Karte ist ein
kompakter Vektor der unmaskierten Pixel; wir bauen die dichte 2D-Karte über die
Maske wieder auf (genau wie `_MapDataset.__getitem__`).

In [ ]:
def reconstruct(compact_row, mask):
    """compact (n_unmasked,) -> dichte 2D-Karte (H, W), 0 in maskierten Pixeln."""
    dense = np.zeros(mask.shape, dtype=np.float32)
    dense[mask] = np.asarray(compact_row, dtype=np.float32)
    return dense

def noise_sigma(ng=NG, pix=PIX_ARCMIN):
    # Shape-Noise sigma = 0.4 / sqrt(2 * ng * pixel_arcmin^2)  (vgl. dataset.py)
    return 0.4 / np.sqrt(2.0 * ng * pix ** 2)

def add_shape_noise(dense, mask, rng):
    noise = rng.standard_normal(dense.shape).astype(np.float32) * noise_sigma()
    return dense + noise * mask          # Noise nur in gültigen Pixeln

print(f"Shape-Noise sigma = {noise_sigma():.4f}")

In [ ]:
# Die Maske selbst.
plt.figure(figsize=(2.8, 9))
plt.imshow(mask, cmap="gray", aspect="auto", interpolation="nearest")
plt.title("Maske (1424×176)")
plt.xlabel("x"); plt.ylabel("y")
plt.colorbar(label="gültig (1) / maskiert (0)", fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Beispielkarten für mehrere Kosmologien, sortiert nach S8 (niedrig -> hoch).
order = np.argsort(S8)
picks = order[np.linspace(0, NCOSMO - 1, 5).astype(int)]

fig, axes = plt.subplots(1, len(picks), figsize=(13, 8))
for ax, c in zip(axes, picks):
    m = reconstruct(kappa[c, 0], mask)
    vmax = np.percentile(np.abs(m[mask]), 99)
    ax.imshow(m, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="nearest")
    ax.set_title(f"$\\Omega_m$={Om[c]:.2f}\n$S_8$={S8[c]:.2f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Beispiel-κ-Karten (rauschfrei), nach $S_8$ sortiert", y=0.94)
plt.tight_layout()
plt.show()

In [ ]:
# Effekt des Shape-Noise: dieselbe Karte rauschfrei vs. mit Noise (wie im Training).
c = int(picks[len(picks) // 2])
rng = np.random.default_rng(0)
clean = reconstruct(kappa[c, 0], mask)
noisy = add_shape_noise(clean, mask, rng)

fig, axes = plt.subplots(1, 2, figsize=(7.5, 9))
vmax = np.percentile(np.abs(clean[mask]), 99)
for ax, img, t in [(axes[0], clean, "rauschfrei"), (axes[1], noisy, "mit Shape-Noise")]:
    ax.imshow(img, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto", interpolation="nearest")
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"κ-Karte — $\\Omega_m$={Om[c]:.2f}, $S_8$={S8[c]:.2f}", y=0.94)
plt.tight_layout()
plt.show()

In [ ]:
# Pixelwert-Verteilung (nur unmaskierte Pixel).
plt.figure(figsize=(7, 4))
plt.hist(clean[mask], bins=120, alpha=0.7, label="rauschfrei", color="steelblue")
plt.hist(noisy[mask], bins=120, alpha=0.5, label="mit Noise", color="indianred")
plt.yscale("log")
plt.xlabel("κ"); plt.ylabel("Anzahl Pixel (log)")
plt.title("Pixelwert-Verteilung")
plt.legend()
plt.tight_layout()
plt.show()

---
### Optional: dieselben Daten über die Projekt-Pipeline laden

Zum Gegenchecken — liefert genau das, was das Modell im Training sieht
(standardisiert, mit Noise). Benötigt die `uv`-Umgebung (torch etc.).

In [ ]:
# from pathlib import Path
# import sys
# sys.path.insert(0, str(Path.cwd().parent))      # Repo-Root auf den Pfad
# from src.dataset import WeakLensingDataset
#
# ds = WeakLensingDataset(data_dir=DATA_DIR).create()
# x, y = ds["train_input"][0]
# print("Sample-Input:", x.shape, "  Label (standardisiert):", y)   # (1,1424,176), (2,)
# print("label_stats (mean, std):", ds["label_stats"])